## SAR Imagery Import

In [ ]:
# Option 1: Box download
import torch
import pandas as pd
import numpy as np
import zipfile
import requests
from pathlib import Path
import os

BOX_URLS = {
    "os1": "https://virginia.box.com/shared/static/9tz0fm7kq6745qzgy22hia985uf6c3hz.zip",
    "os2": "https://virginia.box.com/shared/static/t5dt2l6dgc3oz8oeizj83e8938gx69yr.zip",
    "fs":  "https://virginia.box.com/shared/static/u1lmcljm3o75gseghv5acnd4417qrkn2.zip",
}

PROJECT_ROOT = Path("..").resolve()
DATA_DIR     = PROJECT_ROOT / "data" / "classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_and_extract(key: str, out_dir: Path) -> None:
    url = BOX_URLS[key]
    if not url:
        print(f"[{key}] No URL provided, skipping.")
        return
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[{key}] Already extracted, skipping.")
        return
    zip_path = DATA_DIR / f"{key}.zip"
    print(f"[{key}] Downloading...")
    dl_url = url + ("?dl=1" if "?" not in url else "&dl=1")
    with requests.get(dl_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"[{key}] Extracting...")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf_data:
        zf_data.extractall(out_dir)
    zip_path.unlink()
    print(f"[{key}] Done -> {out_dir}")


download_and_extract("os1", DATA_DIR / "os1")
download_and_extract("os2", DATA_DIR / "os2")
download_and_extract("fs",  DATA_DIR / "fs")

opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

In [ ]:
# Option 2: Google Drive download
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import gdown
import os

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19jLMSzHChVLk-vVAg2muNN2OALzksWob"

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "classification"

if not (DATA_DIR / "opensar1_labels.csv").exists():
    print("Downloading classification folder from Google Drive...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
    print("Download complete.")
else:
    print(f"Data already present at {DATA_DIR}, skipping download.")

opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

## Label Mapping and Filtering

In [ ]:
unique_labels = sorted(set(opensar1['label']).union(
                       set(opensar2['label'])).union(
                       set(fusar['label'])))

label_to_int = {label: i for i, label in enumerate(unique_labels)}

opensar1.loc[:, 'label_id'] = opensar1['label'].map(label_to_int)
opensar2.loc[:, 'label_id'] = opensar2['label'].map(label_to_int)
fusar.loc[:, 'label_id']    = fusar['label'].map(label_to_int)

opensar1 = opensar1[opensar1['label'] != 'unknown']
opensar2 = opensar2[opensar2['label'] != 'unknown']
fusar    = fusar[fusar['label'] != 'unknown']

class_names = [k for k, v in sorted(label_to_int.items(), key=lambda x: x[1]) if k != 'unknown']
print("Label mapping:", label_to_int)
print("class_names:", class_names)

## Dataset Creation

In [ ]:
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torchvision import transforms
from PIL import Image


class create_dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        try:
            image = Image.open(row['path']).convert('L')
        except (FileNotFoundError, OSError) as e:
            raise RuntimeError(f"Error loading image at index {idx}: {row['path']}") from e
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row['label_id'], dtype=torch.long)


RESNET_MEAN = [0.485, 0.456, 0.406]
RESNET_STD  = [0.229, 0.224, 0.225]

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

datasets_val = {
    'os1'  : create_dataset(opensar1, transform=transform_val),
    'os2'  : create_dataset(opensar2, transform=transform_val),
    'fusar': create_dataset(fusar,    transform=transform_val),
}

print(f"OpenSARShip 1: {len(datasets_val['os1'])} samples")
print(f"OpenSARShip 2: {len(datasets_val['os2'])} samples")
print(f"FuSARShip:     {len(datasets_val['fusar'])} samples")

## Reconstruct Test Loaders from Saved Indices

In [ ]:
import zipfile as zf_mod

# ── Extract indices from zip ─────────────────────────────────────────────────
indices_zip = Path("misc/baseline_indices.zip")
with zf_mod.ZipFile(indices_zip, "r") as zf_in:
    zf_in.extractall("misc/")

domain_keys = ['os1', 'os2', 'fusar', 'all']
domain_indices = {}
for key in domain_keys:
    npz = np.load(f"misc/{key}_baseline_indices.npz")
    domain_indices[key] = {
        'train_idx': npz['train_idx'].tolist(),
        'val_idx':   npz['val_idx'].tolist(),
        'test_idx':  npz['test_idx'].tolist(),
    }
    Path(f"misc/{key}_baseline_indices.npz").unlink()

print("Indices loaded:")
for key, idx in domain_indices.items():
    print(f"  {key}: train={len(idx['train_idx'])}  val={len(idx['val_idx'])}  test={len(idx['test_idx'])}")

# ── Rebuild domains + test loaders ──────────────────────────────────────────
opensar1['source'] = 'os1'
opensar2['source'] = 'os2'
fusar['source']    = 'fusar'

domains = {
    'os1'  : {'df': opensar1},
    'os2'  : {'df': opensar2},
    'fusar': {'df': fusar},
    'all'  : {'df': pd.concat([opensar1, opensar2, fusar], ignore_index=True)},
}

full_val_ds = ConcatDataset(list(datasets_val.values()))

for key in domains:
    test_idx = domain_indices[key]['test_idx']
    test_ds  = Subset(full_val_ds if key == 'all' else datasets_val[key], test_idx)

    nw = 0 if key == 'fusar' else 2
    pw = key != 'fusar'
    domains[key]['test_loader'] = DataLoader(
        test_ds, batch_size=32, shuffle=False,
        num_workers=nw, pin_memory=pw, persistent_workers=pw
    )
    print(f"{key}: {len(test_idx)} test samples, {len(domains[key]['test_loader'])} batches")

## Model Imports and Config

In [ ]:
import sys
sys.path.insert(0, str(Path('.').resolve()))

from models.resnet_attention import resnet_attention
from models.resnet_transfer import CNN_resnet_transfer
from models.swin_transfer import swin_transfer

MODEL_CONFIGS = {
    'bam': {'cls': resnet_attention,    'kwargs': {'num_classes': len(class_names)}},
    'res': {'cls': CNN_resnet_transfer, 'kwargs': {'num_classes': len(class_names)}},
    'vis': {'cls': swin_transfer,       'kwargs': {'num_classes': len(class_names)}},
}

## Evaluation

In [ ]:
def evaluate_test(model, label, test_loader, class_names):
    device = next(model.parameters()).device
    model.eval()

    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc        = 100. * correct / total
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    per_class = {}
    print(f"  [{label}] Overall: {acc:.2f}% ({correct}/{total})")
    for i, name in enumerate(class_names):
        mask = all_labels == i
        if mask.sum() == 0:
            continue
        n_correct = int((all_preds[mask] == i).sum())
        n_total   = int(mask.sum())
        cls_acc   = 100. * n_correct / n_total
        per_class[name] = {'acc': round(cls_acc, 4), 'correct': n_correct, 'total': n_total}
        print(f"    {name:15s}: {cls_acc:.2f}% ({n_correct}/{n_total})")

    return acc, per_class


In [ ]:
import gc

# ── Extract weights from zip ─────────────────────────────────────────────────
weights_zip = Path("weights/baseline_weights.zip")
with zf_mod.ZipFile(weights_zip, "r") as zf_in:
    zf_in.extractall("weights/")
print(f"Weights extracted from {weights_zip}")

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = {}  # results[tag][train_domain][test_domain] = {'overall': acc, 'per_class': {...}}

for tag, cfg in MODEL_CONFIGS.items():
    results[tag] = {}
    print(f"\n{'='*60}")
    print(f"  Model: {tag.upper()}")
    print(f"{'='*60}")

    for train_domain in domains:
        weight_path = Path(f"weights/{train_domain}_{tag}_baseline.pth")
        if not weight_path.exists():
            print(f"WARNING: {weight_path.name} not found, skipping")
            continue

        gc.collect()
        torch.cuda.empty_cache()

        model = cfg['cls'](**cfg['kwargs']).to(device)
        model.load_state_dict(torch.load(weight_path, map_location=device))
        print(f"\nLoaded: {weight_path.name}")

        results[tag][train_domain] = {}
        for test_domain in domains:
            acc, per_class = evaluate_test(
                model, f"{train_domain} -> {test_domain}",
                domains[test_domain]['test_loader'], class_names
            )
            results[tag][train_domain][test_domain] = {
                'overall':   round(acc, 4),
                'per_class': per_class,
            }

        model.cpu()
        del model
        gc.collect()
        torch.cuda.empty_cache()

# ── Remove extracted weight files ────────────────────────────────────────────
for tag in MODEL_CONFIGS:
    for domain in domains:
        pth = Path(f"weights/{domain}_{tag}_baseline.pth")
        if pth.exists():
            pth.unlink()

print("\nExtracted weights removed.")


## Results

In [ ]:
import matplotlib.pyplot as plt

domain_keys  = list(domains.keys())
train_colors = ['#7F77DD', '#1D9E75', '#D85A30', '#888780']

for tag in MODEL_CONFIGS:
    if tag not in results or not results[tag]:
        continue

    trained_on = list(results[tag].keys())
    x     = np.arange(len(domain_keys))
    n     = len(trained_on)
    width = 0.8 / n

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.suptitle(f"{tag.upper()} — Cross-Domain Test Accuracy", fontsize=13)

    for i, train_domain in enumerate(trained_on):
        accs    = [results[tag][train_domain].get(t, {}).get('overall', 0) for t in domain_keys]
        offsets = x + i * width - (n - 1) * width / 2
        bars    = ax.bar(offsets, accs, width,
                         label=f"trained on {train_domain}",
                         color=train_colors[i], linewidth=0)
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f"{acc:.1f}", ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(domain_keys)
    ax.set_xlabel("Test set")
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e0e0e0', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
import json

results_path = Path("misc/baseline_test_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

results_zip = Path("misc/baseline_test_results.zip")
with zf_mod.ZipFile(results_zip, "w", zf_mod.ZIP_DEFLATED) as zf_out:
    zf_out.write(results_path, results_path.name)
results_path.unlink()

print(f"Results saved to {results_zip}")
print("Keys:", list(results.keys()))